# M3 · Agentic RAG — MacGyver Gym Rat

**SI4006 · Universidad EAFIT · Module 3** · runs end to end on a free Colab T4.

The M1 model invents exercise names and never refuses. This notebook gives it the
catalog to read, in four steps that each change one thing, and measures every
step with the M2 harness, unchanged:

| Row | What changes | Where it lives |
|---|---|---|
| baseline (M2) | nothing: the M1 LoRA, no retrieval | `eval/harness.py` |
| naive RAG (S07) | dense top-k from Chroma + escape valve | `m3/rag.py` |
| advanced RAG (S08) | hybrid BM25 + dense, cross-encoder rerank, HyDE | `m3/rag.py` |
| final (S10) | a catalog lookup tool on top of the advanced RAG | `m3/agent.py` |

Same eval set, same rubric v1.0, same judge, same generator, same decoding. RAGAS
(`m3/ragas_m3.py`) then looks inside the naive, advanced and final systems.

**Second run (prompt v2).** The first run (`notebooks/M3_rag_v1.ipynb`, results in
`reports/*_m3_v1*`) refused all 14 eval cases: the augmented prompt showed the exact
refusal sentence the M1 LoRA was trained on, right before the request, and the model
copied it into every reply. Section 1 picks a replacement prompt on M1's validation
split, not on the eval set, and **stops the notebook** if no candidate fixes the
collapse, so a failed attempt costs minutes rather than the full hour. Every RAG row
then uses the chosen prompt; nothing else changes.

**Runtime → Change runtime type → T4 GPU.** A full run takes roughly an hour.

**Testing before pushing.** If `m3/` is not in the clone yet, the setup cell asks
for `m3.zip` (the `m3/` folder zipped from the repo root). Set
`M3_ZIP_SOURCE = "drive"` to read it from Google Drive instead of uploading.

## 0 · Setup

In [1]:
import os, sys, subprocess, zipfile
from pathlib import Path

REPO = "https://github.com/iamcroody/models-for-exercises-dataset.git"
BRANCH = "main"
ROOT = Path("/content/models-for-exercises-dataset")

# Only used when m3/ is not in the clone yet: "upload" opens a file picker,
# "drive" reads DRIVE_ZIP from Google Drive.
M3_ZIP_SOURCE = "upload"
DRIVE_ZIP = "/content/drive/MyDrive/m3.zip"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--recurse-submodules", "-b", BRANCH,
                        REPO, str(ROOT)], check=True)
    os.chdir(ROOT)
else:
    ROOT = Path.cwd() if (Path.cwd() / "eval").exists() else Path.cwd().parent
    os.chdir(ROOT)


def unpack_m3(zip_path):
    # Some Windows zip tools write "m3\corpus.py"; Linux would read that as one
    # file name, so separators are normalised before writing anything.
    with zipfile.ZipFile(zip_path) as zf:
        members = [(m, m.filename.replace("\\", "/")) for m in zf.infolist()]
        members = [(m, n) for m, n in members
                   if not n.startswith("__MACOSX") and "__pycache__" not in n]
        target = ROOT if any(n.startswith("m3/") for _, n in members) else ROOT / "m3"
        for member, name in members:
            if name.endswith("/"):
                continue
            path = target / name
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(zf.read(member))


if (ROOT / "m3/corpus.py").exists():
    print("m3/ found in the clone")
elif not IN_COLAB:
    raise RuntimeError("m3/ is missing next to eval/")
else:
    if M3_ZIP_SOURCE == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        zip_path = Path(DRIVE_ZIP)
    else:
        from google.colab import files
        uploaded = files.upload()
        zip_path = Path(next(iter(uploaded)))
    unpack_m3(zip_path)
    if not (ROOT / "m3/corpus.py").exists():
        raise RuntimeError(f"{zip_path} does not contain m3/corpus.py: zip the m3/ "
                         "folder itself, from the repo root")
    print("m3/ unpacked from", zip_path)

print("working in", Path.cwd())
if not (Path.cwd() / "data/exercises-dataset/data/exercises.json").exists():
    raise RuntimeError("the dataset submodule is missing: "
                     "git submodule update --init --recursive")

Saving m3.zip to m3.zip
m3/ unpacked from m3.zip
working in /content/models-for-exercises-dataset


In [2]:
# Exact versions, so a rerun builds the same index and the same table. torchao
# is removed for the same reason as in M1 and M2: Colab pins it to its torch
# build and peft refuses it.
%pip install -q -r m3/requirements.txt
%pip uninstall -y -q torchao

import importlib.metadata as md
VERSIONS = {p: md.version(p) for p in
            ("torch", "transformers", "peft", "sentence-transformers", "chromadb", "rouge_score")}
print(VERSIONS)
print("if Colab asks you to restart the runtime, restart and re-run from the first cell")

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.2 MB/s eta 0:00:00
   ━━━━

In [3]:
import gc, json, csv, statistics
import torch
from transformers import set_seed

for path in ("scripts", "eval", "m3"):
    if path not in sys.path:
        sys.path.insert(0, path)

import macgyver as mg
import harness as H
import corpus as C
import rag as R
import agent as A
import ragas_m3 as G
import probe as P
import selftest

set_seed(H.SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| rubric:", H.load_rubric()["version"], "| seed:", H.SEED)

REPORTS = Path("reports")
REPORTS.mkdir(exist_ok=True)


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

device: cuda | rubric: v1.0 | seed: 42


In [4]:
# Parser, tool, step cap, valve, prompt and corpus checks, with no model weights.
selftest.main()

corpus
  ok   one document per catalog exercise  (1324)
  ok   document ids are unique
  ok   reachable count matches the harness filter  (407)
  ok   every document carries its name and its steps
       {'source': 'https://github.com/jayounghoyos/exercises-dataset (submodule data/exercises-dataset)', 'licence': 'MIT for the exercise metadata and text; images and GIFs (c) Gym visual, not used', 'documents': 1324, 'chunks': 1324, 'reachable_documents': 407, 'chars_min': 352, 'chars_median': 616.0, 'chars_max': 1059, 'words_median': 104.0, 'words_p95': 134, 'words_max': 178, 'targets': 19, 'equipment_classes': 28}
eval set vs corpus
  ok   at least two cases have no valid answer in the corpus  (adv01-impossible-pair, adv03-out-of-domain, adv04-unsafe-object)
  ok   every answerable case has a valid document
  ok   every gold exercise is among the valid documents
  ok   request_text drops the format block  (I want to train my biceps but I have no gym equipment. All I have is two bricks.
R

## 1 · Choosing the prompt (the notebook stops here if no version works)

Three prompts go through the naive RAG on 12 rows of M1's **validation** split
(8 answerable, 4 impossible pairs), none of which shares a muscle and object with the
eval set, so the eval set plays no part in the choice:

| version | what it is |
|---|---|
| `v1` | the first run's prompt, as a control: it should reproduce the collapse |
| `v2a` | same layout, the escape valve said in prose, without the refusal sentence |
| `v2b` | instructions first, then the entries, then the request; no refusal sentence |

The rule is fixed in `m3/probe.py` before anything runs. A version fixes the collapse
if it answers at least half of the answerable rows **and** refuses at least one of the
impossible ones; the best such v2 version is used for every RAG row. If neither does,
the cell stops the notebook. Retrieval here uses the first run's embedding model and
k; section 2 measures both again and checks that they agree.

In [5]:
PROBE_EMBED, PROBE_K = "BAAI/bge-small-en-v1.5", 8

eval_set = H.load_eval_set()
docs = C.build_documents()
BY_ID = {d["id"]: d for d in docs}

P.ensure_splits()
probe_rows = P.dev_rows(eval_set)
print(f"{len(probe_rows)} validation rows:",
      {k: sum(r["kind"] == k for r in probe_rows) for k in ("answerable", "refusal")})

probe_index = R.DenseIndex(docs, PROBE_EMBED, device=DEVICE)
probe_retriever = R.Retriever(docs, probe_index, k=PROBE_K)

tok = mg.load_tokenizer()
model = mg.load_model(mg.BASE_MODEL, adapter="models/r16-all-linear")
llm = H.make_hf_system(model, tok)

probe_replies, probe_scores = {}, {}
for version in (P.CONTROL, *P.CANDIDATES):
    run = R.make_rag(probe_retriever, llm, prompt_version=version)
    probe_replies[version] = [run(r["prompt"])["reply"] for r in probe_rows]
    probe_scores[version] = P.score(probe_replies[version], probe_rows)
    print(f"  {version} done")
del run, llm, model, tok
free_gpu()

print(f"\n{'row':46}" + "".join(f"{v:>12}" for v in probe_replies))
for i, r in enumerate(probe_rows):
    marks = ["refused" if mg.parse_reply(probe_replies[v][i])["refused"] else "answered"
             for v in probe_replies]
    print(f"{r['kind'][:4]} {r['target'][:16]:16} {r['object'][:24]:24}"
          + "".join(f"{m:>12}" for m in marks))

print(f"\n{'version':8}{'answered (answerable)':>24}{'refused (impossible)':>23}  verdict")
for v, s in probe_scores.items():
    print(f"{v:8}{s['answered']:>24.0%}{s['refused']:>23.0%}  {P.verdict(s)}"
          + ("  (control)" if v == P.CONTROL else ""))

PROMPT_VERSION = P.choose(probe_scores)
PROBE = {"rows": probe_rows, "replies": probe_replies, "scores": probe_scores,
         "rule": {"min_answered": P.MIN_ANSWERED, "min_refused": P.MIN_REFUSED},
         "embedding_model": PROBE_EMBED, "top_k": PROBE_K, "chosen": PROMPT_VERSION}

if PROMPT_VERSION is None:
    print("\n" + "=" * 78)
    print("NO v2 PROMPT FIXES THE COLLAPSE. The notebook stops here, before the hour-long")
    print("run. Download this notebook as it is (File -> Download .ipynb): the table above")
    print("is the evidence for the next decision.")
    print("=" * 78)
    raise RuntimeError("no v2 prompt fixes the valve collapse on the validation probe")

print("\n" + "=" * 78)
print(f"PROMPT {PROMPT_VERSION} FIXES THE COLLAPSE on the validation probe: it answers "
      f"{probe_scores[PROMPT_VERSION]['answered']:.0%} of the answerable rows and refuses "
      f"{probe_scores[PROMPT_VERSION]['refused']:.0%} of the impossible ones.")
print(f"Every RAG row below uses prompt {PROMPT_VERSION}.")
print("=" * 78)

12 validation rows: {'answerable': 8, 'refusal': 4}


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

  v1 done
  v2a done
  v2b done

row                                                     v1         v2a         v2b
answ triceps          a bare hallway               refused     refused    answered
answ abs              just the floor and a wal     refused    answered    answered
answ abs              a patch of grass in the      refused    answered    answered
answ upper back       two full paint cans          refused     refused    answered
answ delts            a backpack held by one s     refused    answered    answered
answ cardiovascular s a patch of grass in the      refused    answered    answered
answ biceps           two one-litre juice cart     refused    answered    answered
answ abs              nothing at all               refused    answered    answered
refu adductors        two full paint cans          refused     refused    answered
refu triceps          a filled shopping bag        refused    answered    answered
refu spine            a duffel bag full of lau     ref

## 2 · Corpus and index

**Corpus:** the catalog itself, `data/exercises-dataset` pinned at commit
`8872272`. Exercise metadata and text are MIT; the images and GIFs (© Gym visual)
are not used.

**Chunk = one exercise, no overlap.** Dimension 3 grades the exercise *name* against
the catalog and ROUGE-L grades the *steps* against that exercise's steps, so the name
and its steps must reach the generator together. The statistics below show that every
record is short enough to stay whole.

In [6]:
stats = C.corpus_stats(docs)
for k, v in stats.items():
    print(f"  {k:22} {v}")

VALID = {e["id"]: C.valid_ids_for(e, docs) for e in eval_set}
print("\ncases with no valid answer in the corpus:",
      [e["id"] for e in eval_set if not VALID[e["id"]]])
print("valid documents per answerable case:",
      {e["id"]: len(VALID[e["id"]]) for e in eval_set if VALID[e["id"]]})

  source                 https://github.com/jayounghoyos/exercises-dataset (submodule data/exercises-dataset)
  licence                MIT for the exercise metadata and text; images and GIFs (c) Gym visual, not used
  documents              1324
  chunks                 1324
  reachable_documents    407
  chars_min              352
  chars_median           616.0
  chars_max              1059
  words_median           104.0
  words_p95              134
  words_max              178
  targets                19
  equipment_classes      28

cases with no valid answer in the corpus: ['adv01-impossible-pair', 'adv03-out-of-domain', 'adv04-unsafe-object']
valid documents per answerable case: {'gold01-forearms': 2, 'gold02-glutes': 1, 'gold03-calves': 1, 'gold04-upper-back': 5, 'gold05-biceps': 29, 'gold06-delts': 15, 'gold07-abs': 63, 'gold08-triceps': 10, 'gold09-pectorals': 3, 'gold10-levator-scapulae': 2, 'adv02-false-premise': 29}


### Decision · embedding model

Two candidates: the harness's multilingual MiniLM and an English-only BGE model,
because the corpus is English. Two measurements decide it: how many documents each
model has to **truncate** (an exercise whose last steps fall off the end is embedded
without them), and **hit@k**, the share of cases with a valid answer for which at least
one valid exercise reaches the top-k, using the raw prompt as the query exactly as the
naive RAG will.

In [7]:
KS = (1, 3, 5, 8, 10)
SCORED = [e for e in eval_set if VALID[e["id"]]]


def hit_at(search, query_of, ks=KS):
    ranked = {e["id"]: search(query_of(e)) for e in SCORED}
    return {k: round(sum(bool(set(ranked[e["id"]][:k]) & VALID[e["id"]]) for e in SCORED)
                     / len(SCORED), 3) for k in ks}


indexes, embedding_report = {}, {}
for model_id in R.EMBEDDING_CANDIDATES:
    index = probe_index if model_id == PROBE_EMBED else R.DenseIndex(docs, model_id, device=DEVICE)
    tok = index.encoder.tokenizer
    limit = index.encoder.max_seq_length
    lengths = [len(tok(d["text"])["input_ids"]) for d in docs]
    hits = hit_at(lambda q: index.search(q, max(KS)), lambda e: e["input"])
    embedding_report[model_id] = {
        "max_seq_length": limit,
        "docs_truncated": sum(n > limit for n in lengths),
        "tokens_median": statistics.median(lengths),
        "tokens_max": max(lengths),
        "hit_at_k_raw_prompt": hits,
    }
    indexes[model_id] = index
    print(model_id, json.dumps(embedding_report[model_id], indent=1))

harness_model = next(iter(R.EMBEDDING_CANDIDATES))
EMBED_MODEL = max(R.EMBEDDING_CANDIDATES,
                  key=lambda m: (embedding_report[m]["hit_at_k_raw_prompt"][5], m == harness_model))
print("\nchosen embedding model:", EMBED_MODEL)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (144 > 128). Running this sequence through the model will result in indexing errors


sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 {
 "max_seq_length": 128,
 "docs_truncated": 1202,
 "tokens_median": 158.0,
 "tokens_max": 272,
 "hit_at_k_raw_prompt": {
  "1": 0.182,
  "3": 0.182,
  "5": 0.182,
  "8": 0.182,
  "10": 0.273
 }
}
BAAI/bge-small-en-v1.5 {
 "max_seq_length": 512,
 "docs_truncated": 0,
 "tokens_median": 138.0,
 "tokens_max": 234,
 "hit_at_k_raw_prompt": {
  "1": 0.091,
  "3": 0.364,
  "5": 0.364,
  "8": 0.455,
  "10": 0.455
 }
}

chosen embedding model: BAAI/bge-small-en-v1.5


### Decision · top-k

The smallest k that already reaches the best hit rate available within the first ten
results. Past that point more documents only add tokens the 1.7B generator has to read
around.

In [8]:
dense = indexes.pop(EMBED_MODEL)
for other in list(indexes):
    del indexes[other]
free_gpu()

hits = embedding_report[EMBED_MODEL]["hit_at_k_raw_prompt"]
TOP_K = min(k for k in KS if k >= 3 and hits[k] >= hits[max(KS)])
words = stats["words_median"]
print("hit@k:", hits)
print(f"TOP_K = {TOP_K}: about {TOP_K * words:.0f} words of context per prompt "
      f"({words:.0f} words per median exercise)")

if (EMBED_MODEL, TOP_K) == (PROBE_EMBED, PROBE_K):
    print("the prompt probe used these same decisions")
else:
    print(f"WARNING: the prompt probe used {PROBE_EMBED} with k={PROBE_K}; "
          "this run chose differently, so the probe's retrieval is not this run's")

reranker = R.load_reranker(device=DEVICE)
retriever = R.Retriever(docs, dense, reranker=reranker, k=TOP_K, pool=20)
DECISIONS = {"chunk": "one exercise per chunk, no overlap", "embedding_model": EMBED_MODEL,
             "top_k": TOP_K, "rerank_pool": retriever.pool, "reranker": R.RERANKER_MODEL,
             "embedding_report": embedding_report, "prompt_version": PROMPT_VERSION,
             "prompt_probe": PROBE}

hit@k: {1: 0.091, 3: 0.364, 5: 0.364, 8: 0.455, 10: 0.455}
TOP_K = 8: about 832 words of context per prompt (104 words per median exercise)
the prompt probe used these same decisions


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

## 3 · Generating every system

The free T4 cannot hold the generator, the judge and the RAGAS judge together, so,
as in M2, every reply is generated first and graded afterwards through `H.replay`.
Each system is wrapped in a `Recorder`: the harness still sees `prompt → reply`, and
the latency, retrieved contexts and tool trace are kept on the side.

Latency is measured per query over the whole system call (retrieval, HyDE draft,
tool steps, final generation) after one warm-up generation, and reported as the median.

In [9]:
RECORDERS = {}


def warm_up(llm):
    llm(eval_set[0]["input"])


def run_system(name, run):
    rec = R.Recorder(name, run, BY_ID)
    for i, e in enumerate(eval_set, 1):
        rec(e["input"])
        print(f"  {name:16} {i:2}/{len(eval_set)}  {rec.log[e['input']]['latency_s']:6.1f}s",
              end="\r", flush=True)
    print(f"  {name:16} done, median {rec.median_latency():.1f}s per query" + " " * 10)
    RECORDERS[name] = rec
    return rec


def release_recorders():
    """Every Recorder closes over llm, and llm over the model: drop them first."""
    for rec in RECORDERS.values():
        rec.run = None

### Baseline rows (no retrieval): zero-shot and the M1 LoRA

In [10]:
tok = mg.load_tokenizer()
model = mg.load_model(mg.BASE_MODEL)
llm = H.make_hf_system(model, tok)
warm_up(llm)
run_system("zeroshot", R.make_baseline(llm))
release_recorders()
del model, tok, llm
free_gpu()

tok = mg.load_tokenizer()
model = mg.load_model(mg.BASE_MODEL, adapter="models/r16-all-linear")
llm = H.make_hf_system(model, tok)
warm_up(llm)
run_system("lora_m1", R.make_baseline(llm))

m2 = json.loads(Path("reports/replies_m2.json").read_text())
for name, key in (("zeroshot", "zeroshot"), ("lora_m1", "lora")):
    same = sum(RECORDERS[name].log[e["input"]]["reply"] == m2[key][e["input"]] for e in eval_set)
    print(f"{name}: {same}/{len(eval_set)} replies identical to the M2 run")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

  zeroshot         done, median 5.4s per query          


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

  lora_m1          done, median 7.9s per query          
zeroshot: 14/14 replies identical to the M2 run
lora_m1: 14/14 replies identical to the M2 run


### Naive RAG (S07)

Dense top-k with the raw prompt as the query, the catalog entries prepended, and the
escape valve, in the prompt version section 1 chose. The valve is not shown as a
sentence any more: the LoRA's own trained refusal starts with `No safe option: I know
no ...`, which is the phrase the frozen harness parser recognises.

In [11]:
run_system("rag_naive", R.make_rag(retriever, llm, prompt_version=PROMPT_VERSION))

print(R.augment(eval_set[4]["input"], retriever.docs_for(
    RECORDERS["rag_naive"].log[eval_set[4]["input"]]["context_ids"]),
    PROMPT_VERSION)[:1600], "...")

  rag_naive        done, median 7.7s per query          
Catalog entries that may be relevant:

[1] Exercise: biceps narrow pull-ups
Target muscle: biceps
Equipment: body weight
Body part: upper arms
Secondary muscles: forearms, shoulders
Steps:
1. Hang from a pull-up bar with your palms facing towards you and your hands shoulder-width apart.
2. Engage your core and pull yourself up towards the bar, focusing on using your biceps to lift your body.
3. Pause for a moment at the top, then slowly lower yourself back down to the starting position.
4. Repeat for the desired number of repetitions.

[2] Exercise: dumbbell alternate biceps curl (with arm blaster)
Target muscle: biceps
Equipment: dumbbell
Body part: upper arms
Secondary muscles: forearms
Steps:
1. Stand up straight with your feet shoulder-width apart and hold a dumbbell in each hand, palms facing forward.
2. Place the arm blaster on your upper arms, ensuring a secure fit.
3. Keeping your upper arms stationary, exhale and curl on

### Diagnosis: where does the naive RAG fail, and which technique attacks it?

This cell uses retrieval only, no generation (HyDE reuses the LoRA's no-RAG reply as
its draft, the same text the live system regenerates). For every case: did a valid
exercise reach the top-k under each technique? Together with the generated naive
replies, this sorts the naive failures into **retrieval** (no valid exercise in the
context), **generation** (it was there and the model ignored it) and **out of corpus**
(there is nothing valid to retrieve).

In [12]:
draft = lambda p: RECORDERS["lora_m1"].log[p]["reply"]
VARIANTS = {"naive": (), "hybrid": ("hybrid",), "rerank": ("rerank",), "hyde": ("hyde",),
            "hybrid+rerank": ("hybrid", "rerank"), "all three": R.TECHNIQUES}

retrieval_hits = {}
print(f"{'case':26}" + "".join(f"{v:>14}" for v in VARIANTS))
for e in eval_set:
    row = []
    for v, techniques in VARIANTS.items():
        ids, _ = retriever.retrieve(e["input"], techniques, draft)
        found = len(set(ids) & VALID[e["id"]])
        retrieval_hits.setdefault(v, []).append(found > 0 if VALID[e["id"]] else None)
        row.append("n/a" if not VALID[e["id"]] else f"{found}/{TOP_K}")
    print(f"{e['id']:26}" + "".join(f"{c:>14}" for c in row))

print("\nhit@k over cases with a valid answer:")
for v, flags in retrieval_hits.items():
    flags = [f for f in flags if f is not None]
    print(f"  {v:14} {sum(flags)}/{len(flags)}")

case                               naive        hybrid        rerank          hyde hybrid+rerank     all three
gold01-forearms                      0/8           0/8           0/8           0/8           0/8           0/8
gold02-glutes                        0/8           0/8           0/8           0/8           0/8           0/8
gold03-calves                        0/8           0/8           0/8           0/8           0/8           0/8
gold04-upper-back                    0/8           0/8           0/8           4/8           0/8           4/8
gold05-biceps                        3/8           2/8           4/8           8/8           3/8           6/8
gold06-delts                         0/8           0/8           0/8           0/8           0/8           0/8
gold07-abs                           7/8           3/8           7/8           2/8           7/8           6/8
gold08-triceps                       1/8           0/8           0/8           6/8           0/8           4/8
g

### Advanced RAG (S08) and the ablation

The advanced system stacks the three techniques. Each attacks one failure the cell
above can show:

| technique | failure it attacks |
|---|---|
| hybrid (BM25 + dense, RRF) | exact catalog terms the embedding blurs: a rare muscle name, the equipment word |
| rerank (cross-encoder over a pool of 20) | the right muscle in the top-k, but with the wrong equipment class first |
| HyDE (request + the LoRA's own draft as the query) | "two bricks" never says "dumbbell"; the draft does, in the catalog's own wording |

Ablation: the advanced system with one technique removed at a time, on the same
eval set, so each technique's contribution and cost can be read off directly.

In [13]:
ADVANCED = R.TECHNIQUES
run_system("rag_advanced", R.make_rag(retriever, llm, ADVANCED, PROMPT_VERSION))
for removed in ADVANCED:
    kept = tuple(t for t in ADVANCED if t != removed)
    run_system(f"rag_adv_no_{removed}", R.make_rag(retriever, llm, kept, PROMPT_VERSION))

  rag_advanced     done, median 15.2s per query          
  rag_adv_no_hybrid done, median 14.7s per query          
  rag_adv_no_rerank done, median 14.9s per query          
  rag_adv_no_hyde  done, median 7.8s per query          


### Final system (S10): the catalog lookup tool

`find_exercises(target, equipment)` returns the reachable catalog exercises for one
muscle and one equipment class, or says that there are none. The model proposes the
call as JSON; `agent.parse_tool_call` accepts the usual small-model damage (code
fences, single quotes, trailing commas, prose around it, function-call syntax) and
validates the arguments against the schema's enums. At most `A.MAX_STEPS` calls, and
a repeated call ends the loop.

In [14]:
print(json.dumps(A.tool_schema({d["target"] for d in docs}), indent=1)[:900], "...")
print("max steps:", A.MAX_STEPS)

run_system("rag_final", A.make_agent(retriever, llm, docs, ADVANCED,
                                     prompt_version=PROMPT_VERSION))

replies_out = {name: rec.log for name, rec in RECORDERS.items()}
(REPORTS / "replies_m3.json").write_text(json.dumps(replies_out, indent=2, ensure_ascii=False))
print("wrote reports/replies_m3.json")

{
 "name": "find_exercises",
 "description": "Look up real catalog exercises for one target muscle and one equipment class. Returns up to five exercises with their steps, or says that the catalog has none.",
 "parameters": {
  "target": {
   "type": "string",
   "enum": [
    "abductors",
    "abs",
    "adductors",
    "biceps",
    "calves",
    "cardiovascular system",
    "delts",
    "forearms",
    "glutes",
    "hamstrings",
    "lats",
    "levator scapulae",
    "pectorals",
    "quads",
    "serratus anterior",
    "spine",
    "traps",
    "triceps",
    "upper back"
   ],
   "description": "the muscle the user wants to train"
  },
  "equipment": {
   "type": "string",
   "enum": [
    "dumbbell",
    "kettlebell",
    "weighted",
    "body weight"
   ],
   "description": "the class the user's object stands in for: dumbbell = a matched pair, one per hand; kettlebell = one load ...
max steps: 3
  rag_final        done, median 38.9s per query          
wrote reports/replies_m3

In [15]:
# Clean delta: retrieval does not depend on the prompt, so the naive and advanced
# rows must have retrieved exactly what the first run retrieved. The first run's
# file is only in the clone once it has been pushed; otherwise this is skipped.
V1_REPLIES = REPORTS / "replies_m3_v1.json"
if V1_REPLIES.exists():
    v1 = json.loads(V1_REPLIES.read_text(encoding="utf-8"))
    for name in ("rag_naive", "rag_advanced", "rag_adv_no_hybrid", "rag_adv_no_rerank",
                 "rag_adv_no_hyde"):
        same = sum(RECORDERS[name].log[e["input"]]["context_ids"]
                   == v1[name][e["input"]]["context_ids"] for e in eval_set)
        print(f"{name:18} {same}/{len(eval_set)} retrievals identical to the first run")
else:
    print("reports/replies_m3_v1.json is not in the clone: retrieval check against the "
          "first run skipped")

reports/replies_m3_v1.json is not in the clone: retrieval check against the first run skipped


In [16]:
# Free the generator. The closures go first, or empty_cache runs with the
# weights still referenced.
release_recorders()
del model, tok, llm
free_gpu()
print("GPU memory in use:", round(torch.cuda.memory_allocated() / 2**30, 2) if DEVICE == "cuda" else 0, "GB")

GPU memory in use: 0.22 GB


## 4 · The M2 harness on every row

Same `harness(eval_set, system)`, same judge (`Qwen/Qwen2.5-1.5B-Instruct`), same
rubric v1.0, same embeddings. The systems are replayed from the recorded replies.

In [17]:
judge = H.Judge(device=DEVICE)
similarity = H.Similarity()

SCORECARDS = {}
for name, rec in RECORDERS.items():
    print(f"\n{name}")
    SCORECARDS[name] = H.harness(eval_set, H.replay(rec.replies()), judge=judge,
                                 similarity=similarity, label=name)

H.print_scorecard(list(SCORECARDS.values()))
H.write_scorecard(list(SCORECARDS.values()), REPORTS / "scorecard_m3.csv")

with (REPORTS / "latency_m3.csv").open("w", newline="", encoding="utf-8") as fh:
    writer = csv.writer(fh)
    writer.writerow(["system", "median_latency_s", "mean_latency_s", "max_latency_s", "n"])
    for name, rec in RECORDERS.items():
        values = [r["latency_s"] for r in rec.log.values()]
        writer.writerow([name, rec.median_latency(), round(statistics.mean(values), 3),
                         max(values), len(values)])

(REPORTS / "scorecard_m3_detail.json").write_text(json.dumps(
    {"scorecards": SCORECARDS, "decisions": DECISIONS, "versions": VERSIONS,
     "advanced_techniques": ADVANCED, "max_steps": A.MAX_STEPS},
    indent=2, ensure_ascii=False, default=str))
print("\nwrote scorecard_m3.csv, latency_m3.csv and scorecard_m3_detail.json")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


zeroshot
  [ 1/14] gold01-forearms            miss sim=0.72 judge=5
  [ 2/14] gold02-glutes              miss sim=0.66 judge=5
  [ 3/14] gold03-calves              miss sim=0.71 judge=5
  [ 4/14] gold04-upper-back          miss sim=0.75 judge=5
  [ 5/14] gold05-biceps              miss sim=0.79 judge=5
  [ 6/14] gold06-delts               miss sim=0.62 judge=5
  [ 7/14] gold07-abs                 miss sim=0.79 judge=5
  [ 8/14] gold08-triceps             miss sim=0.73 judge=5
  [ 9/14] gold09-pectorals           miss sim=0.74 judge=5
  [10/14] gold10-levator-scapulae    miss sim=0.71 judge=5
  [11/14] adv01-impossible-pair      miss sim=0.59 judge=5
  [12/14] adv02-false-premise        miss sim=0.65 judge=5
  [13/14] adv03-out-of-domain        miss sim=0.60 judge=5
  [14/14] adv04-unsafe-object        miss sim=0.64 judge=5

lora_m1
  [ 1/14] gold01-forearms            miss sim=0.86 judge=5
  [ 2/14] gold02-glutes              miss sim=0.65 judge=5
  [ 3/14] gold03-calves              

In [18]:
EVOLUTION = [("Baseline M2 (no RAG)", "lora_m1"), ("Naive RAG (S07)", "rag_naive"),
             ("Advanced RAG", "rag_advanced"), ("Final (with tool)", "rag_final")]


def fmt(v, pct=False):
    return "n/a" if v is None else (f"{v:.1%}" if pct else f"{v:.3f}")


print("| System | Dim 1 similarity | Dim 2 judge | Dim 3 hit rate | Median latency |")
print("|---|---:|---:|---:|---:|")
for label, name in EVOLUTION + [("Zero-shot (reference)", "zeroshot")] + [
        (f"Advanced without {t}", f"rag_adv_no_{t}") for t in ADVANCED]:
    sc = SCORECARDS[name]
    print(f"| {label} | {fmt(sc['similarity_mean'])} | {fmt(sc['judge_mean'])} | "
          f"{fmt(sc['domain_hit_rate'], True)} | {RECORDERS[name].median_latency():.1f} s |")

| System | Dim 1 similarity | Dim 2 judge | Dim 3 hit rate | Median latency |
|---|---:|---:|---:|---:|
| Baseline M2 (no RAG) | 0.787 | 4.714 | 21.4% | 7.9 s |
| Naive RAG (S07) | 0.656 | 3.214 | 7.1% | 7.7 s |
| Advanced RAG | 0.705 | 2.429 | 42.9% | 15.2 s |
| Final (with tool) | 0.704 | 3.000 | 28.6% | 38.9 s |
| Zero-shot (reference) | 0.694 | 5.000 | 0.0% | 5.4 s |
| Advanced without hybrid | 0.731 | 2.214 | 42.9% | 14.7 s |
| Advanced without rerank | 0.664 | 2.071 | 14.3% | 14.9 s |
| Advanced without hyde | 0.662 | 3.571 | 14.3% | 7.8 s |


## 5 · Where it fails, and what the valve and the tool did

In [19]:
for name in ("lora_m1", "rag_naive", "rag_advanced", "rag_final"):
    sc = SCORECARDS[name]
    print(f"\n=== {name}: {sc['domain_hits']}/{sc['n']} domain hits ===")
    for e, row in zip(eval_set, sc["detail"]):
        ids = RECORDERS[name].log[e["input"]]["context_ids"]
        kind = R.classify_failure(e, row, ids, VALID[e["id"]])
        if kind != "ok":
            reply = " ".join(row["reply"].split())[:150]
            print(f"  {e['id']:26} {kind}\n      {row['why_missed']} | {reply}")


=== lora_m1: 3/14 domain hits ===
  gold01-forearms            no retrieval
      named an exercise that is not in the catalog | Exercise: dumbbell reverse curl Gym equivalent: dumbbell Adaptation: Use two filled water bottles in place of the dumbbells, one in each hand. Match t
  gold02-glutes              no retrieval
      the exercise needs equipment the object cannot stand in for | Exercise: kettlebell goblet squat Gym equivalent: kettlebell Adaptation: Use a backpack loaded with books as the kettlebell. Grip it firmly and keep y
  gold03-calves              no retrieval
      named an exercise that is not in the catalog | Exercise: kettlebell calf raise Gym equivalent: kettlebell Adaptation: Use a filled shopping bag in place of the kettlebell. Grip it firmly and keep y
  gold04-upper-back          no retrieval
      named an exercise that is not in the catalog | Exercise: dumbbell standing row Gym equivalent: dumbbell Adaptation: Use two bags of rice in place of the dumbbells, 

In [20]:
print("The escape valve on the adversarial cases\n")
adversarial = [e for e in eval_set if e["adversarial"]]
print(f"{'case':26}" + "".join(f"{n:>14}" for n in ("lora_m1", "rag_naive", "rag_advanced", "rag_final")))
for e in adversarial:
    cells = []
    for name in ("lora_m1", "rag_naive", "rag_advanced", "rag_final"):
        row = next(r for r in SCORECARDS[name]["detail"] if r["id"] == e["id"])
        refused = mg.parse_reply(row["reply"])["refused"]
        cells.append(("hit" if row["hit"] else "miss") + (" (valve)" if refused else ""))
    print(f"{e['id']:26}" + "".join(f"{c:>14}" for c in cells))

The escape valve on the adversarial cases

case                             lora_m1     rag_naive  rag_advanced     rag_final
adv01-impossible-pair               miss          miss   hit (valve)   hit (valve)
adv02-false-premise                 miss   hit (valve)   hit (valve)   hit (valve)
adv03-out-of-domain                 miss          miss          miss          miss
adv04-unsafe-object                 miss          miss   hit (valve)          miss


In [21]:
def tool_needed(example, ids):
    """Needed when the retrieved entries hold no valid exercise for a mapped object.

    That includes the impossible pair, where only the tool can confirm the absence.
    Not needed for the dosing question or the curtain rail: neither is a lookup.
    """
    if example["meta"].get("equipment") is None:
        return False
    return not (set(ids) & VALID[example["id"]])


log = RECORDERS["rag_final"].log
retrieved_k = {e["id"]: log[e["input"]]["context_ids"][:TOP_K] for e in eval_set}
print(f"{'case':26}{'needed':>8}{'called':>8}  calls")
summary = {"needed_and_called": 0, "needed_not_called": 0, "not_needed_called": 0,
           "not_needed_not_called": 0}
for e in eval_set:
    trace = log[e["input"]]["trace"]
    calls = A.tool_calls(trace)
    needed = tool_needed(e, retrieved_k[e["id"]])
    if needed:
        summary["needed_and_called" if calls else "needed_not_called"] += 1
    else:
        summary["not_needed_called" if calls else "not_needed_not_called"] += 1
    shown = "; ".join(f"{c['args']['target']}/{c['args']['equipment']}->{len(c['results'])}"
                      for c in calls)
    other = [t["action"] for t in trace if t["action"] not in ("tool_call", "answer")]
    print(f"{e['id']:26}{str(needed):>8}{len(calls):>8}  {shown} {other if other else ''}")
print("\n", summary)

case                        needed  called  calls
gold01-forearms               True       1  forearms/body weight->3 ['repeat_call']
gold02-glutes                 True       0   
gold03-calves                 True       1  calves/body weight->5 ['repeat_call']
gold04-upper-back            False       1  upper back/body weight->5 ['repeat_call']
gold05-biceps                False       1  biceps/body weight->1 ['repeat_call']
gold06-delts                  True       1  delts/dumbbell->5 ['repeat_call']
gold07-abs                   False       0   
gold08-triceps               False       1  triceps/body weight->5 ['repeat_call']
gold09-pectorals             False       1  pectorals/body weight->5 ['repeat_call']
gold10-levator-scapulae      False       0   
adv01-impossible-pair         True       1  lats/dumbbell->0 ['repeat_call']
adv02-false-premise          False       0   ['parse_error', 'parse_error', 'parse_error', 'step_cap']
adv03-out-of-domain          False       1  glutes/b

## 6 · RAGAS

The four metrics as the S10 lab computes them, with two corrections that matter for
a small judge: an unparseable verdict is `None` and counted (never a default 3), and
context precision and recall follow the RAGAS definitions (rank-weighted precision,
per-sentence recall). The judge is `Qwen/Qwen2.5-3B-Instruct`, loaded only now that
the generator and the harness judge are gone; if it does not fit, `load_judge` falls
back to the 1.5B model and the table says which one graded.

In [22]:
del judge
free_gpu()

ragas_judge, fallback_notes = G.load_judge(device=DEVICE)
print("RAGAS judge:", ragas_judge.model_id, "| fallback notes:", fallback_notes or "none")

RAGAS, RAGAS_ROWS = {}, {}
for name in ("rag_naive", "rag_advanced", "rag_final"):
    print(f"\n{name}")
    cases = G.build_cases(eval_set, RECORDERS[name].log)
    RAGAS[name], RAGAS_ROWS[name] = G.evaluate(ragas_judge, similarity, cases)

with (REPORTS / "ragas_m3.csv").open("w", newline="", encoding="utf-8") as fh:
    writer = csv.writer(fh)
    writer.writerow(["system", *G.METRICS, "unparsed_verdicts", "judge_model", "n"])
    for name, s in RAGAS.items():
        writer.writerow([name, *[s[m] for m in G.METRICS], s["unparsed_verdicts"],
                         s["judge_model"], len(eval_set)])
(REPORTS / "ragas_m3_detail.json").write_text(json.dumps(
    {"summary": RAGAS, "per_case": RAGAS_ROWS, "fallback_notes": fallback_notes},
    indent=2, ensure_ascii=False))

print("\n| System | Faithfulness | Answer relevancy | Context precision | Context recall |")
print("|---|---:|---:|---:|---:|")
for name, s in RAGAS.items():
    print(f"| {name} | " + " | ".join(fmt(s[m]) for m in G.METRICS) + " |")
print("\nunparsed verdicts:", {n: s["unparsed_verdicts"] for n, s in RAGAS.items()})

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

RAGAS judge: Qwen/Qwen2.5-3B-Instruct | fallback notes: none

rag_naive
  gold01-forearms            faithf=0.71 answer=0.81 contex=0.00 contex=0.27
  gold02-glutes              faithf=0.60 answer=0.60 contex=0.48 contex=0.36
  gold03-calves              faithf=0.00 answer=0.85 contex=0.20 contex=0.50
  gold04-upper-back          faithf=0.82 answer=0.64 contex=0.12 contex=0.18
  gold05-biceps              faithf=0.00 answer=0.76 contex=0.00 contex=0.73
  gold06-delts               faithf=0.70 answer=0.63 contex=0.39 contex=0.30
  gold07-abs                 faithf=0.60 answer=0.55 contex=0.94 contex=0.70
  gold08-triceps             faithf=0.80 answer=0.63 contex=1.00 contex=0.27
  gold09-pectorals           faithf=0.75 answer=0.61 contex=0.14 contex=0.10
  gold10-levator-scapulae    faithf=0.00 answer=0.65 contex=0.50 contex=0.36
  adv01-impossible-pair      faithf=0.70 answer=0.60 contex=0.00 contex=0.00
  adv02-false-premise        faithf=0.00 answer=0.76 contex=0.00 contex=0.69
  ad

## 7 · Bring the results back

Downloads every `reports/*_m3*` file of this run as one zip (the first run's
`*_m3_v1*` files are left out). Unzip it at the repo root, then save
this executed notebook (File → Download .ipynb) over `notebooks/M3_rag.ipynb`.

In [23]:
bundle = Path("m3_results.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(p for p in REPORTS.glob("*_m3*") if "_v1" not in p.name):
        zf.write(path, path.as_posix())
        print("  ", path)
if IN_COLAB:
    from google.colab import files
    files.download(str(bundle))

   reports/latency_m3.csv
   reports/ragas_m3.csv
   reports/ragas_m3_detail.json
   reports/replies_m3.json
   reports/scorecard_m3.csv
   reports/scorecard_m3_detail.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>